In [ ]:
'''
new_vaccinations:
O gün uygulanan yeni toplam doz sayısıdır.

new_vaccinations_smoothed:
Günlük yeni dozların 7 günlük hareketli ortalamasıdır.

people_vaccinated_per_hundred:
Her 100 kişiden en az bir doz aşı olan kişi sayısıdır.

people_fully_vaccinated_per_hundred:
Her 100 kişiden tam aşılı kişi sayısıdır.

total_boosters_per_hundred:
Her 100 kişi başına uygulanan toplam hatırlatma dozu sayısıdır.

KULLANIM KURALLARI

total_vaccinations ve total_boosters doz sayısını gösterir.
Bir kişinin birden fazla dozu bulunabileceği için bu sütunlar kişi sayısı olarak yorumlanmaz.

Ülkeler arası karşılaştırmalarda people_vaccinated_per_hundred
ve people_fully_vaccinated_per_hundred kullanılacaktır.

Aşılama sütunları kümülatif olduğunda tarihler boyunca toplanmayacaktır.
Seçilen tarihteki veya haftanın son gözlemindeki hazır değer kullanılacaktır.
'''

'\nAŞI VERİLERİ\ntotal_vaccinations\tToplam yapılan doz sayısı (bir kişi 2 doz olduysa 2 sayılır)\npeople_vaccinated\tEn az 1 doz olan kişi sayısı\npeople_fully_vaccinated\tTam aşılı (2 doz vb.) kişi sayısı\ntotal_boosters\tYapılan hatırlatma (booster) dozu sayısı\n*_per_hundred uzantılı\tYüz kişi başına oran\n'

In [1]:
#gerekli kütüphaneleri ve temiz veriyi yüklüyoruz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_countries = pd.read_csv(
    r'C:\Users\nazlican\OneDrive\Masaüstü\CoronaVirusProject\data\processed\countries_clean.csv',
    parse_dates=['date']
)

print("Satır-sütun sayısı:", df_countries.shape)
print("Ülke sayısı:", df_countries['location'].nunique())
print("İlk tarih:", df_countries['date'].min())
print("Son tarih:", df_countries['date'].max())

print(
    "Kalan ülke-tarih tekrarı:",
    df_countries.duplicated( #aynı satır var mı kontrol ediyoruz
        subset=['location', 'date']
    ).sum()
)

Satır-sütun sayısı: (324819, 67)
Ülke sayısı: 194
İlk tarih: 2020-01-01 00:00:00
Son tarih: 2024-08-14 00:00:00
Kalan ülke-tarih tekrarı: 0


In [3]:
#veri setindeki değerlerden kaç tane veri girişi yapılmış sayıyoruz
vaccination_coverage = df_countries.groupby('location').agg(
    toplam_satir=('date', 'size'),
    toplam_doz_kaydi=('total_vaccinations', 'count'),
    ilk_doz_kaydi=('people_vaccinated', 'count'),
    tam_asi_kaydi=('people_fully_vaccinated', 'count'),
    hatirlatma_dozu_kaydi=('total_boosters', 'count'),
    ilk_doz_orani_kaydi=(
        'people_vaccinated_per_hundred',
        'count'
    ),
    tam_asi_orani_kaydi=(
        'people_fully_vaccinated_per_hundred',
        'count'
    )
)

vaccination_coverage['bos_tam_asi_orani_kaydi'] = (
    vaccination_coverage['toplam_satir']
    - vaccination_coverage['tam_asi_orani_kaydi']
)

print(
    vaccination_coverage
    .sort_values('tam_asi_orani_kaydi')
    .head(15)
)

                        toplam_satir  toplam_doz_kaydi  ilk_doz_kaydi  \
location                                                                
Eritrea                         1674                 0              0   
Marshall Islands                1674                 0              0   
Micronesia (country)            1674                 0              0   
North Korea                     1674                 0              0   
Liechtenstein                   1674               420            420   
Vatican                         1674                 0              0   
Switzerland                     1674               893            893   
Palau                           1674                 0              0   
Turkmenistan                    1674                 5              4   
Tuvalu                          1674                12             12   
Nauru                           1674                18             18   
Ethiopia                        1674               

In [4]:
#people_fully_vaccinated_per_hundred kaydı olan ülkelerde,
#boşlukların başta mı, arada mı, sonda mı olduğunu inceliyoruz
partial_full_vaccination_missing = vaccination_coverage[
    (vaccination_coverage['tam_asi_orani_kaydi'] > 0)
    & (vaccination_coverage['bos_tam_asi_orani_kaydi'] > 0)
].copy()

missing_full_vaccination_rows = df_countries.loc[
    df_countries['location'].isin(
        partial_full_vaccination_missing.index
    )
    & df_countries[
        'people_fully_vaccinated_per_hundred'
    ].isna(),
    [
        'location',
        'date',
        'people_fully_vaccinated_per_hundred'
    ]
].copy()

valid_full_vaccination_range = (
    df_countries.loc[
        df_countries[
            'people_fully_vaccinated_per_hundred'
        ].notna()
    ]
    .groupby('location')
    .agg(
        ilk_dolu_tam_asi_tarihi=('date', 'min'),
        son_dolu_tam_asi_tarihi=('date', 'max')
    )
)

missing_full_vaccination_positions = (
    missing_full_vaccination_rows
    .join(
        valid_full_vaccination_range,
        on='location'
    )
)

missing_full_vaccination_positions['bosluk_konumu'] = 'arada'

missing_full_vaccination_positions.loc[
    missing_full_vaccination_positions['date']
    < missing_full_vaccination_positions[
        'ilk_dolu_tam_asi_tarihi'
    ],
    'bosluk_konumu'
] = 'başta'

missing_full_vaccination_positions.loc[
    missing_full_vaccination_positions['date']
    > missing_full_vaccination_positions[
        'son_dolu_tam_asi_tarihi'
    ],
    'bosluk_konumu'
] = 'sonda'

full_vaccination_missing_position_summary = (
    missing_full_vaccination_positions
    .groupby(['location', 'bosluk_konumu'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['başta', 'arada', 'sonda'], fill_value=0)
)

print(full_vaccination_missing_position_summary)

bosluk_konumu  başta  arada  sonda
location                          
Afghanistan      492    825    217
Albania          394    732    329
Algeria          593    358    700
Andorra          428    878    315
Angola           493    867    217
...              ...    ...    ...
Venezuela        523    508    625
Vietnam          481    519    401
Yemen            554    842    238
Zambia           502    502    440
Zimbabwe         442     75    665

[186 rows x 3 columns]


In [5]:
print(
    "Kısmen eksik tam aşılama oranı bulunan ülke sayısı:",
    len(full_vaccination_missing_position_summary)
)

print(
    "Arada eksik tam aşılama oranı bulunan ülke sayısı:",
    (
        full_vaccination_missing_position_summary['arada']
        .gt(0)
        .sum()
    )
)

print(
    "Toplam arada eksik kayıt sayısı:",
    full_vaccination_missing_position_summary['arada'].sum()
)

Kısmen eksik tam aşılama oranı bulunan ülke sayısı: 186
Arada eksik tam aşılama oranı bulunan ülke sayısı: 179
Toplam arada eksik kayıt sayısı: 88718


In [7]:
#anormal veriler var mı kontrol ediyoruz
vaccination_value_checks = pd.Series({
    'Negatif toplam doz kaydı': ( #0 dan küçük mü
        df_countries['total_vaccinations']
        .lt(0)
        .sum()
    ),
    'Negatif ilk doz kişi kaydı': (
        df_countries['people_vaccinated']
        .lt(0)
        .sum()
    ),
    'Negatif tam aşılı kişi kaydı': (
        df_countries['people_fully_vaccinated']
        .lt(0)
        .sum()
    ),
    'Negatif hatırlatma dozu kaydı': (
        df_countries['total_boosters']
        .lt(0)
        .sum()
    ),
    'Tam aşılı kişi sayısının ilk doz kişi sayısını geçtiği kayıt': (
        (
            df_countries['people_fully_vaccinated']
            > df_countries['people_vaccinated']
        )
        .sum()
    )
})

print(vaccination_value_checks)

Negatif toplam doz kaydı                                        0
Negatif ilk doz kişi kaydı                                      0
Negatif tam aşılı kişi kaydı                                    0
Negatif hatırlatma dozu kaydı                                   0
Tam aşılı kişi sayısının ilk doz kişi sayısını geçtiği kayıt    0
dtype: int64
